## Short intorduction on VLM

FineVisionMax is a dataset designed for training open vision-language models (VLMs) via direct streaming so no need to download the whole dataset and train the model. Only samples with single image are used in this notebook for reference, but in real VLM/VLA more than one image is usually used, but for simplicity the single image is adopted.

The VLM contains from vision model and LLM, both of them trained separately on massive pretrained datataset. The cheapest way is just train VLM is using the image project layer, gated cross-attention layers and tahn gating parameters like it is done in Flamingo Visual Language Model:
![flamingo](https://miro.medium.com/v2/resize:fit:1400/1*R71aXteo6vUYIkEI30diUg.png)

But most of the model is not using this training style, but append visual token and embeddings tokens together and pass into Large Language Model:
![QwenVL](The-training-pipeline-of-the-Qwen-VL-series.png)

There are several stages for training VLM (like LLaVA):
1. Feature Aligment Pre-training At the first stage the vison model projections is trained so the Vision Model would be able to have a better connection with LLM and the LLM would provide correct output based on vision projected features. Both the Vision Encoder and the LLM are frozen.
2. End2End Visual Instruction Tuning: After the projection is sucessfully trained the projection and the LLM model together on the specific tasks and usually the vision model is not trained since we do not want to corrupt the vision model and its vision features extraction capabilities. The Vision Model remains frozen.

Adantages:
1. It is cheaper than training QwenVL like model.
2. It fully preserves the original pre-trained vision representations.

Disadvantages:
1. The vision model is bottelneck since it might be able to work properly with OCR-text
2. It is not be able to work properly with precise coordinates (grounding)

In order to solve the issue above QwenLV has the following training stages:
1. Large-Scale Vision-Language Pre-training: training the ViT encoder and the projection layer using a massive corpus of image-text pairs, unlike LLaVA which projection layer is trained. The LLM remains frozen.
2. Multi-Task Joint Training: the ViT, adapter and LLM are fully trained so the model can resolve issue with fine-grained perception (like reading small text or finding pixel coordinates). The model is training to solve QA, OCR and task grounding etc.
3. Supervised Instruction Fine-Tuning (SFT): the model alined with chat behavior, multi-turn dialogues, and human preferences. The ViT is frozen or LoRA is used during SFT. 

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
print(next(iter(dataset)))

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoProcessor, AutoTokenizer, AutoModelForCausalLM

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder_ckpt, lm_ckpt, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        self.vision_encoder = AutoModel.from_pretrained(vision_encoder_ckpt).vision_model
        self.vision_processor = AutoProcessor.from_pretrained(vision_encoder_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim)
        self.tokenizer = AutoTokenizer.from_pretrained(lm_ckpt)
        self.llm = AutoModelForCausalLM.from_pretrained(lm_ckpt)

    def forward(self, text, image, labels=None):
        processed_image = self.vision_processor(images=[image], return_tensors="pt").to(device=self.llm.device)
        image_embed = self.vision_encoder(**processed_image).last_hidden_state
        image_embed = self.modality_projector(image_embed).to(dtype=self.llm.dtype)

        input_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(device=self.llm.device)
        token_embed = self.llm.model.embed_tokens(input_ids)

        combined = torch.cat([image_embed, token_embed], dim=1)

        logits = self.llm(inputs_embeds=combined).logits

        shift_logits = logits[:, image_embed.size(1):-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        loss = nn.functional.cross_entropy(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))

        return logits, loss

In [ ]:
vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"

vlm = VisionLanguageModel(vision_encoder_ckpt, language_model_ckpt).to("cuda")

In [ ]:
from transformers.image_utils import load_image

image = load_image("https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000285.jpg")
text = "What does this say?"
output = vlm(text, image)

In [ ]:
image

In [ ]:
output

In [ ]:
sample = next(iter(dataset))

def apply_chat_template(sample):
    message_list = []
    for data_dict in sample["texts"]:
        message_list.append({
            'role': 'user', 'content': data_dict["user"]
        })
        message_list.append({
            'role': 'assistant', 'content': data_dict["assistant"]
        })
    text = vlm.tokenizer.apply_chat_template(message_list, tokenize=False)
    return text

text = apply_chat_template(sample)

In [ ]:
sample['texts'][0]

In [ ]:
sample['images'][0]

In [ ]:
text

In [ ]:
vlm(text, sample["images"])

## Simple training

At the beggining it is fine to fine-tune the VLM like this but it has several flaws: the training is done with single batch and context window is not fully used, but at least it clear what is happening during the training VLM. But it should be mentioned that there are several issues that ending token and padding is the same so we will not be able to stop untill hit the limit for token generation, missing gradient accumulation (batch is too small), the visual projection tokens are also used for prediction.

In [ ]:
import json
import torch.optim as optim

opt = optim.AdamW(vlm.parameters(), lr=1e-5)

step = 0
losses = []
for sample in dataset:
    text = apply_chat_template(sample)
    if len(text) > 3000 or len(sample["images"]) != 1:
        continue

    images = [sample["images"][0].convert("RGB")]
    step += 1

    opt.zero_grad()
    logits, loss = vlm(text, images)

    loss.backward()
    opt.step()

    losses.append(loss.detach().item())

    if step % 100 == 0:
        print(f"step {step} | loss {loss:.4f}")

    if step > 5000:
      json.dump([round(l, 4) for l in losses], open("chapter_3_loss.json", "w"))
      break

In [ ]:
logits, loss = vlm(text, images)

In [ ]:
tokens = torch.argmax(logits, dim=-1)

In [ ]:
tokens

In [ ]:
text

In [ ]:
print(''.join(vlm.tokenizer.batch_decode(tokens)))

## Packed batch (Sample Packing or Multipack)

In order to use the packed batch we also have to use the 4D mask which include data related to sample segments so we could build the proper attention mask, positional embedding for text input and correctly calculate the loss without image token, predicting `<bos>` based on the previous segemnt's token `<eos>` and correctly hadling `<eos>` with `<pad>`.

```
Tokens:  [Img_1, Prompt_1, Ans_1, Img_2, Prompt_2, Ans_2]
Segment: [--- Sample 1 --------] [--- Sample 2 --------]

```

### 4D input masking
Since the batch can contain severals samples we would like to have a proper masking since that might cause the data leakage if the information for the current and previous batch might be correlated. 
The visualization of batch packing in order to fully utilize the context window of the model without too many padding tokens which are not used in loss compute. 
![Batch packing](https://huggingface.co/datasets/vlmbook/notebooks/resolve/main/Chapter%203/balanced_knapsack_packing.png)

In [ ]:
def build_packed_allowed_mask(padding_mask, segment_ids):
    """
    Args:
       padding_mask (B, T) - mask with valid token and '0' for padding
       segment_ids (B, T) - contains the ids for each segment to set proper mask
    """
    if padding_mask.ndim != 2 and padding_mask.shape != segment_ids.shape:
        raise ValueError("padding_mask and segment_ids must both have shape [B, T]")
    if not torch.all((padding_mask == 0) | (padding_mask == 1)):
        raise ValueError("padding_mask values must be 0 or 1")

    valid = padding_mask.bool()
    batch_size, sequence_length = valid.shape
    device = valid.device

    causal = torch.tril(
        torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=device)
    ).unsqueeze(0)
    same_segment = segment_ids.unsqueeze(-1).eq(segment_ids.unsqueeze(1))
    valid_key = valid.unsqueeze(1)
    allowed = causal & same_segment & valid_key
    return allowed.unsqueeze(1)

def format_packed_attention_mask(allowed, activation_dtype=torch.float32, attention_backend="sdpa"):
    if attention_backend not in {"sdpa", "eager"}:
        raise ValueError("Use attention_backend='sdpa' or 'eager' for a custom 4D mask")
    bias = torch.zeros(allowed.shape, dtype=activation_dtype, device=allowed.device)
    bias.masked_fill_(~allowed, torch.finfo(activation_dtype).min)
    return bias

In [ ]:
class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder_ckpt, language_model_ckpt, tokenizer, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        self.vision_encoder = AutoModel.from_pretrained(vision_encoder_ckpt).vision_model
        self.vision_processor = AutoProcessor.from_pretrained(vision_encoder_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim, bias=False)
        self.tokenizer = tokenizer
        self.llm = AutoModelForCausalLM.from_pretrained(language_model_ckpt)
        self.llm.resize_token_embeddings(len(self.tokenizer))  # Resize the LLM's token embeddings to match the tokenizer's new vocab size

    def _replace_img_tokens_with_embed(
        self,
        input_ids,
        segment_ids,
        padded_mask,
        token_embed,
        image_embed,
    ):
        updated = token_embed.clone()

        valid_segments = segment_ids[padded_mask.bool()]
        active_segments = torch.unique(valid_segments, sorted=True)

        expected_segments = torch.arange(
            image_embed.size(0),
            device=segment_ids.device,
        )
        if not torch.equal(active_segments, expected_segments):
            raise ValueError(
                "Segment IDs must be contiguous and match the local image order"
            )

        for image_index in range(image_embed.size(0)):
            image_positions = (
                segment_ids.eq(image_index)
                & input_ids.eq(self.tokenizer.image_token_id)
                & padded_mask.bool()
            )
            image_vectors = image_embed[image_index]
            updated[image_positions] = image_vectors.to(updated.dtype)
        return updated

    @staticmethod
    def build_packed_labels(input_ids, padding_mask, segment_ids, image_token_id):
        labels = input_ids.clone()
        valid = padding_mask.bool()

        labels[~valid] = -100
        labels[input_ids == image_token_id] = -100

        segment_start = torch.zeros_like(valid, dtype=torch.bool, device=input_ids.device)
        segment_start[:, 0] = valid[:, 0]
        segment_start[:, 1:] = valid[:, 1:] & segment_ids[:, 1:].ne(segment_ids[:, :-1])
        # segment_ids[:, 1:].ne(segment_ids[:, :-1])
        # we do not want to make prediction based <eos> to <bos>
        labels[segment_start] = -100
        return labels

    def forward(self, input_ids, images, padded_mask, position_ids, segment_ids):
        token_embed = self.llm.model.embed_tokens(input_ids)

        combined = []

        for batch in range(input_ids.size(0)):
            row_images = images[batch]
            processed_image = self.vision_processor(images=row_images, return_tensors="pt").to(self.llm.device)
            image_embed = self.vision_encoder(**processed_image).last_hidden_state
            image_embed = self.modality_projector(image_embed)

            row_combined = self._replace_img_tokens_with_embed(
                input_ids[batch],
                segment_ids[batch],
                padded_mask[batch],
                token_embed[batch],
                image_embed,
            )
            combined.append(row_combined)
        combined = torch.stack(combined, dim=0)

        allowed = build_packed_allowed_mask(
            padded_mask,
            segment_ids
        )
        packed_attention_mask = format_packed_attention_mask(
            allowed=allowed,
            activation_dtype=combined.dtype,
            attention_backend="sdpa"
        )

        logits = self.llm(
            inputs_embeds=combined,
            attention_mask=packed_attention_mask,
            position_ids=position_ids,
            use_cache=False,
        ).logits

        labels = self.build_packed_labels(
            input_ids,
            padded_mask,
            segment_ids,
            self.tokenizer.image_token_id,
        )

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        supervised = shift_labels.ne(-100)
        if not supervised.any():
            raise ValueError("Packed batch has no supervised language targets")

        loss = nn.functional.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.flatten(),
            ignore_index=-100
        )

        return logits, loss

In [ ]:
vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(language_model_ckpt, extra_special_tokens={"image_token": "<|image|>"})
vlm = VisionLanguageModel(vision_encoder_ckpt, language_model_ckpt, tokenizer).to("cuda")

optimizer = optim.AdamW(vlm.parameters(), lr=1e-4)

def tokenize_sample(sample):
    message_list = []
    for data_dict in sample["texts"]:
        message_list.append({
            "role": "user", "content": data_dict['user']
        })
        message_list.append({
            "role": "assistant", "content": data_dict['assistant']
        })
    message_list[0]["content"] = tokenizer.image_token * 256 + message_list[0]['content']
    tokens = tokenizer.apply_chat_template(
        message_list,
        tokenize=True,
        return_dict=True,
    )
    return tokens["input_ids"]

step = 0
losses = []
max_length = 2048

# current sample data
cur_sample = []
cur_image = []
cur_segment_ids = []
cur_position_ids = []

# keep track segments
segment = 0

# keep track batch data
inputs_batch = []
image_batch = []
segment_ids_batch = []
padded_input_batch = []
padded_mask_batch = []
position_ids_batch = []
start = time.time()
max_batch_size = 1
batch_complete = False
for sample in dataset:
    if len(sample["images"]) != 1:
        continue
    tokens = tokenize_sample(sample)
    if len(tokens) > max_length:
        continue
    if len(cur_sample) + len(tokens) <  max_length:
        cur_segment_ids.extend([segment] * len(tokens))
        cur_position_ids.extend(list(range(len(tokens))))
        segment += 1
        cur_sample.extend(tokens)
        cur_image.extend([sample["images"][0].convert("RGB")])
    else:
        sequence_length = len(cur_sample)
        remaining_padding = max_length - sequence_length
        cur_segment_ids.extend([0] * remaining_padding)
        cur_position_ids.extend([0] * remaining_padding)
        padded_mask = torch.full((max_length,), 0, dtype=torch.long)
        padded_mask[:sequence_length] = 1

        padded_input_ids = torch.full((max_length,), tokenizer.pad_token_id, dtype=torch.long)
        padded_input_ids[:sequence_length] = torch.tensor(cur_sample, dtype=torch.long)

        inputs_batch.append(padded_input_ids)
        segment_ids_batch.append(torch.tensor(cur_segment_ids))
        padded_input_batch.append(padded_input_ids)
        padded_mask_batch.append(padded_mask)
        position_ids_batch.append(torch.tensor(cur_position_ids, dtype=torch.long))
        image_batch.append(cur_image)

        cur_sample = []
        cur_image = []
        cur_segment_ids = []
        cur_position_ids = []

        if len(inputs_batch) == max_batch_size:
            batch_complete = True # We've reached the max batch size, so we can continue
        segment = 0
        cur_sample = tokens
        cur_image = [sample['images'][0].convert('RGB')]
        cur_segment_ids = [segment] * len(tokens)
        cur_position_ids = list(range(len(tokens)))
        segment += 1

    if batch_complete:
        padded_mask = torch.stack(padded_mask_batch, dim=0).cuda()
        final_position_ids = torch.stack(position_ids_batch, dim=0).cuda()
        input_ids = torch.stack(inputs_batch, dim=0).cuda()
        segments_ids_tensor = torch.stack(segment_ids_batch, dim=0).cuda()

        step += 1
        optimizer.zero_grad()
        logits, loss = vlm(
            input_ids=input_ids,
            images=image_batch,
            padded_mask=padded_mask,
            position_ids=final_position_ids,
            segment_ids=segments_ids_tensor,
        )
        losses.append(loss.item())

        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            print(f"step {step} | loss {loss.item():.4f} | smoothed loss: {np.mean(losses[-10:]):.4f}")
            print(f"The last 100 steps took: {time.time()-start:.2f} seconds.")
            start = time.time()

        del logits, loss, padded_mask, final_position_ids, input_ids, segments_ids_tensor
        torch.cuda.empty_cache()

        batch_complete = False
        batch = []
        attention_mask_batch = []
        image_batch = []

        inputs_batch = []
        image_batch = []
        segment_ids_batch = []
        padded_input_batch = []
        padded_mask_batch = []
        position_ids_batch = []

        if step > 1000:
            break


In [ ]:
tokenizer.image_token

In [ ]:
# there is not KV-cache so recomputed the whole sequence
from transformers.image_utils import load_image
import torch

image = load_image("https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000002006.jpg")

@torch.no_grad()
def generate(trained_vlm, tokenizer, input_text, image, max_new_tokens):
    text = [{"role": "user", "content": tokenizer.image_token * 256 + input_text}]
    text_str = tokenizer.apply_chat_template(text, add_generation_prompt=True, tokenize=False)

    input_tokens = tokenizer(text_str, return_tensors="pt").input_ids.cuda()  # Shape: (1, T)

    nested_image_input = [[image]]

    output = []
    end_of_sentence_token = tokenizer.eos_token_id

    while len(output) < max_new_tokens:
        current_len = input_tokens.size(1)

        padded_mask = torch.ones((1, current_len), dtype=torch.long, device=input_tokens.device)
        position_ids = torch.arange(current_len, dtype=torch.long, device=input_tokens.device).unsqueeze(0)
        segment_ids = torch.zeros((1, current_len), dtype=torch.long, device=input_tokens.device)

        logits, _ = trained_vlm(
            input_ids=input_tokens,
            images=nested_image_input,
            padded_mask=padded_mask,
            position_ids=position_ids,
            segment_ids=segment_ids
        )

        next_token_tensor = torch.argmax(logits[:, -1, :], dim=-1)  # Shape: (1,)
        next_token_id = next_token_tensor.item()

        if next_token_id == end_of_sentence_token:
            break

        output.append(next_token_id)

        input_tokens = torch.cat([input_tokens, next_token_tensor.unsqueeze(-1)], dim=1)

    return tokenizer.decode(output, skip_special_tokens=True)

vlm.eval()
print(generate(vlm, tokenizer, "What do you see here?", image, 32))
